In [1]:
import os
import numpy as np
import random
import time
import datetime
import json
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.multiprocessing import Pool, set_start_method
import shap
import concurrent.futures
from torch.utils.data import DataLoader
import cuml
from cuml.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from torch.utils.data import Dataset
from PIL import Image
from finetune_data.utils import pre_caption
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from models.blip import blip_decoder
import utils
from utils import cosine_lr_schedule
from finetune_data import create_dataset, create_sampler, create_loader
from finetune_data.utils import save_result, coco_caption_eval

In [2]:
transform = transforms.Compose([
        transforms.Resize((384, 384), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
    ])
class CustomDataset(Dataset):
    def __init__(self, transform, image_root, disturbance_root, raw_data):
        '''
        image_root (string): Root directory of images (e.g. coco/images/)
        ann_root (string): directory to store the annotation file
        disturbance_root (string): directory of disturbance images
        '''
        self.raw_data = raw_data
        self.transform = transform
        self.image_root = image_root
        self.disturbance_root = disturbance_root

    def __len__(self):
        return len(self.raw_data)

    def __getitem__(self, index):

        ann = self.raw_data[index]

        image_path = os.path.join(self.image_root, ann['image'])
        image = Image.open(image_path).convert('RGB')
        image = self.transform(image)

        disturbance_path = os.path.join(self.disturbance_root, ann['image'])
        disturbance_image = Image.open(disturbance_path).convert('RGB')
        disturbance_image = self.transform(disturbance_image) 
        # tokenized_keywords
        tokenized_keywords = ann['tokenized_keywords']

        # return image, disturbance_image, int(ann['image_id']), tokenized_keywords
        return image, disturbance_image, tokenized_keywords
def custom_collate_fn(batch):
    images, disturbance_images, tokenized_keywords_list = zip(*batch)
    images = torch.stack(images, 0)
    disturbance_images = torch.stack(disturbance_images)
    tokenized_keys = [token for sample in tokenized_keywords_list for token in sample]
    return images, disturbance_images, tokenized_keys

In [20]:
# data_path = "../dataset/flickr30k/dbi_val_with_tokenized_keywords.json"
data_path = "../dataset/flickr30k/edu_val_with_tokenized_keywords.json"
# data_path = "../dataset/flickr30k/fi_val_with_tokenized_keywords.json"
# data_path = "../dataset/flickr30k/idc_val_with_tokenized_keywords.json"
# data_path = "../dataset/flickr30k/mi_val_with_tokenized_keywords.json"
# data_path = "../dataset/flickr30k/resume_val_with_tokenized_keywords.json"
image_root = "../dataset/flickr30k/images/"
disturbance_image_root = "../dataset/flickr30k/disturbance_images/"
with open(data_path, 'r') as f:
    data = json.load(f)

dataset = CustomDataset(transform, image_root, disturbance_image_root, data[64:])
dataloader = DataLoader(dataset, batch_size=64, collate_fn=custom_collate_fn)

In [4]:
pretrained_model = [
        "../output/Caption_flickr_sensitive_dbi/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_edu/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_fi/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_idc/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_mi/checkpoint_best.pth",
        "../output/Caption_flickr_sensitive_resume/checkpoint_best.pth"
    ]
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
model = blip_decoder(pretrained=pretrained_model[1], image_size=384, vit='base',
                                  vit_grad_ckpt=False, vit_ckpt_layer=0,
                                  prompt='a picture of ')
model = model.to(device)
model.eval()

load checkpoint from ../output/Caption_flickr_sensitive_edu/checkpoint_best.pth


BLIP_Decoder(
  (visual_encoder): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (blocks): ModuleList(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (act): GELU()
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
      (1): Block

In [5]:
target_layers = list(range(12))

In [6]:
def custom_generate(model, image, sample=False, num_beams=3, max_length=30, min_length=10, top_p=0.9,
                    repetition_penalty=1.0, output_scores=False, return_dict_in_generate=False):

    image_embeds = model.visual_encoder(image)

    if not sample:
        image_embeds = image_embeds.repeat_interleave(num_beams, dim=0)

    image_atts = torch.ones(image_embeds.size()[:-1], dtype=torch.long).to(image.device)

    model_kwargs = {"encoder_hidden_states": image_embeds, "encoder_attention_mask": image_atts}

    prompt = [model.prompt] * image.size(0) 
    input_ids = model.tokenizer(prompt, return_tensors="pt").input_ids.to(image.device)  # tokenize prompt

    input_ids[:, 0] = model.tokenizer.bos_token_id  

    input_ids = input_ids[:, :-1] 

    if sample:
        # nucleus sampling
        outputs = model.text_decoder.generate(input_ids=input_ids,
                                                     max_length=max_length,
                                                     min_length=min_length,
                                                     do_sample=True,
                                                     top_p=top_p,
                                                     num_return_sequences=1,
                                                     eos_token_id=model.tokenizer.sep_token_id,
                                                     pad_token_id=model.tokenizer.pad_token_id,
                                                     repetition_penalty=1.1,
                                                     output_scores=output_scores,
                                                     return_dict_in_generate=return_dict_in_generate,
                                                     **model_kwargs)
        outputs = outputs[:, model.prompt_length:]
    else:
        # beam search
        outputs = model.text_decoder.generate(input_ids=input_ids,
                                                     max_length=max_length,
                                                     min_length=min_length,
                                                     num_beams=num_beams,
                                                     eos_token_id=model.tokenizer.sep_token_id,
                                                     pad_token_id=model.tokenizer.pad_token_id,
                                                     repetition_penalty=repetition_penalty,
                                                     output_scores=output_scores,
                                                     return_dict_in_generate=return_dict_in_generate,
                                                     **model_kwargs)

    return outputs

In [7]:
def compute_unlearning_outputs(model, images, tokenized_keywords, target_layers, device):

    ffn_outputs = []

    def capture_ffn_output(module, input, output):
        ffn_outputs.append(output)

    def register_ffn_hook(model, target_layers):
        hooks = []
        for idx, layer in enumerate(model.text_decoder.bert.encoder.layer):
            if idx in target_layers:
                hook = layer.intermediate.register_forward_hook(capture_ffn_output)
                hooks.append(hook)
        return hooks
    
    hooks = register_ffn_hook(model, target_layers)
    res = custom_generate(model, images, sample=True)
    tokenized_keywords = torch.tensor(tokenized_keywords, device=res.device)

    mask = torch.isin(res, tokenized_keywords)

    unlearning_first_token_outputs = []
    target_layers_len = len(target_layers)
    layer_outputs = [[] for _ in range(target_layers_len)]
    for i in range(len(ffn_outputs)):
        layer_index = (i - target_layers_len) % target_layers_len
        if i < target_layers_len:
            layer_outputs[layer_index].append(ffn_outputs[i][:, -1, :])
            unlearning_first_token_outputs.append(ffn_outputs[i][:, -1, :])
        else:
            layer_outputs[layer_index].append(ffn_outputs[i][:, 0, :])
    layer_outputs = [torch.stack(layer) for layer in layer_outputs]

    unlearning_sensitive_outputs = []
    mask_t = mask.T

    keywords_indices = mask_t.nonzero(as_tuple=True)

    for layer_index in range(target_layers_len):
        layer_output = layer_outputs[layer_index]

        selected_outputs = layer_output[keywords_indices]

        unlearning_sensitive_outputs.append(selected_outputs.view(-1, selected_outputs.size(-1)))

    for hook in hooks:
        hook.remove()
    # torch.cuda.empty_cache()
    return unlearning_first_token_outputs, unlearning_sensitive_outputs

In [8]:
def compute_disturbance_outputs(model, images, target_layers, device):

    ffn_outputs = []

    def capture_ffn_output(module, input, output):
        ffn_outputs.append(output)

    def register_ffn_hook(model, target_layers):
        hooks = []
        for idx, layer in enumerate(model.text_decoder.bert.encoder.layer):
            if idx in target_layers:
                hook = layer.intermediate.register_forward_hook(capture_ffn_output)
                hooks.append(hook)
        return hooks
    
    hooks = register_ffn_hook(model, target_layers)
    _ = custom_generate(model, images, sample=True)

    disturbance_first_token_outputs = []
    for i in range(len(target_layers)):
        disturbance_first_token_outputs.append(ffn_outputs[i][:, -1, :])

    for hook in hooks:
        hook.remove()
    # torch.cuda.empty_cache()
    return disturbance_first_token_outputs

In [9]:
def parameters_input_solver(unlearning_outputs, disturbance_outputs, target_layers, device):

    layer_indices = []

    unlearning_outputs_tensor = torch.stack(unlearning_outputs, dim=0)
    disturbance_outputs_tensor = torch.stack(disturbance_outputs, dim=0)

    scores = torch.mean(torch.abs(unlearning_outputs_tensor - disturbance_outputs_tensor), dim=1).to(device)
    for layer_idx in target_layers:
        if layer_idx <9:
            print(len(torch.where(scores[layer_idx] >= 0.02)[0].tolist()))
            layer_indices.append(torch.where(scores[layer_idx] >= 0.02)[0].tolist())
        else:

            layer_indices.append(torch.topk(scores[layer_idx], 1000).indices.tolist())
    del scores
    return layer_indices

In [10]:
def train_auxiliary_model(model, criterion, optimizer, dataloader, device, num_epochs=15):
    for epoch in range(num_epochs):
        total_loss = 0.0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device).long()

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * inputs.size(0)
        epoch_loss = total_loss / len(dataloader.dataset)
        print(f'Epoch {epoch}/{num_epochs}, Loss: {epoch_loss}')

In [11]:
def evaluate_auxiliary_model(model, dataloader, device):
    model.eval()
    corrects, total = 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device).long()
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels).item()
            total += inputs.size(0)
    accuracy = corrects / total
    print(f'Accuracy: {accuracy}')
    return accuracy

In [12]:
def compute_indices_by_shapley_values(shap_values, layer, device):
    shap_values_tensor = torch.stack([torch.tensor(shap_val, device=device) for shap_val in shap_values], dim=0)[:, :, 0].mean(dim=0)

    if layer < 9:

        print(f"layer {layer} num:", len(torch.where(shap_values_tensor >= 2e-4)[0].tolist()))
        return torch.where(shap_values_tensor >= 2e-4)[0].tolist()
    else:

        return torch.topk(shap_values_tensor, 1000).indices.tolist()

In [13]:
class AuxiliaryNet(nn.Module):
    def __init__(self):
        super(AuxiliaryNet, self).__init__()
        self.fc1 = nn.Linear(3072, 768)
        self.fc2 = nn.Linear(768, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.fc1(x)
        out = self.fc2(self.relu(out))
        return out


def weight_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)


In [14]:
def parameters_shap_solver(unlearning_outputs, disturbance_outputs, device):
    layer_indices = []
    for idx, unlearning_output in enumerate(unlearning_outputs):
        # start_time = time.time()
        disturbance_output = disturbance_outputs[idx]

        dec_data = torch.cat([unlearning_output, disturbance_output], dim=0)
        dec_labels = torch.cat([torch.ones(unlearning_output.shape[0]), torch.zeros(disturbance_output.shape[0])], dim=0)

        dec_train_data, dec_test_data, dec_train_labels, dec_test_labels = train_test_split(dec_data, dec_labels, test_size=0.2, random_state=42)
        dec_train_loader = DataLoader(TensorDataset(dec_train_data, dec_train_labels), batch_size=64, shuffle=True)
        dec_test_loader = DataLoader(TensorDataset(dec_test_data, dec_test_labels), batch_size=64, shuffle=False)

        auxiliary_model = AuxiliaryNet().to(device)
        auxiliary_model.apply(weight_init)
        optimizer = torch.optim.Adam(auxiliary_model.parameters(), lr=1e-4)
        criterion = nn.CrossEntropyLoss()
        train_auxiliary_model(auxiliary_model, criterion, optimizer, dec_train_loader, device, num_epochs=10)
        accuracy = evaluate_auxiliary_model(auxiliary_model, dec_test_loader, device)
        print(f'Layer {idx}, Accuracy: {accuracy}')
        print(dec_train_data.shape[0])
        # print('Training time {}'.format(str(datetime.timedelta(seconds=int(time.time() - start_time)))))
        # start_time = time.time()
        y1 = 100
        indices = torch.randint(0, dec_train_data.shape[0], (y1,))
        sub_dec_train_data = dec_train_data[indices]

        explainer = shap.DeepExplainer(auxiliary_model, sub_dec_train_data)

        print(unlearning_output.shape[0])  # x
        y2 = 100
        indices = torch.randint(0, unlearning_output.shape[0], (y2,))
        sub_unlearning_output = unlearning_output[indices]

        shap_values = explainer.shap_values(sub_unlearning_output)

        layer_indices.append(compute_indices_by_shapley_values(shap_values, idx, device))

        # torch.cuda.empty_cache()

    return layer_indices

In [15]:
def compute_ffn_outputs(model, batch, target_layers, device):

    images, disturbance_images, tokenized_keywords = batch
    images = images.to(device)
    disturbance_images = disturbance_images.to(device)
    unlearning_first_token_outputs, unlearning_sensitive_outputs = compute_unlearning_outputs(
            model, images, tokenized_keywords, target_layers, device
    )
    disturbance_first_token_outputs = compute_disturbance_outputs(
            model, disturbance_images, target_layers, device
    )
    return unlearning_first_token_outputs, unlearning_sensitive_outputs, disturbance_first_token_outputs

In [16]:
def merge_indices(input_indices, shap_indices):

    result = []
    for i in range(len(input_indices)):
        result.append(list(set(input_indices[i]).intersection(set(shap_indices[i]))))
    return result

In [17]:
def get_parameter_indices(model, dataloader, target_layers, device):
    model.eval()
    target_layers_len = len(target_layers)

    unlearning_first_token_output = None
    unlearning_sensitive_output = None
    disturbance_output = None
    for idx, batch in enumerate(dataloader):
      
        unlearning_first_token_output, unlearning_sensitive_output, disturbance_output = compute_ffn_outputs(model, batch, target_layers, device)
       
    # torch.cuda.empty_cache()
    
    input_indices = parameters_input_solver(unlearning_first_token_output, disturbance_output, target_layers, device)
    shap_indices = parameters_shap_solver(unlearning_sensitive_output, disturbance_output, device)
    parameter_indices = merge_indices(input_indices, shap_indices)
    
    return parameter_indices

In [21]:
parameter_indices = get_parameter_indices(model, dataloader, target_layers, device)

99
96
86
115
185
275
247
194
235
Epoch 0/10, Loss: 0.3971692440195829
Epoch 1/10, Loss: 0.1604128519203437
Epoch 2/10, Loss: 0.12016049705047176
Epoch 3/10, Loss: 0.11128030576333096
Epoch 4/10, Loss: 0.1085654865689729
Epoch 5/10, Loss: 0.10782394346631603
Epoch 6/10, Loss: 0.10509942827646625
Epoch 7/10, Loss: 0.10597884010354923
Epoch 8/10, Loss: 0.10357015512485072
Epoch 9/10, Loss: 0.10281189598541692
Accuracy: 0.9426229508196722
Layer 0, Accuracy: 0.9426229508196722
486
544
layer 0 num: 286
Epoch 0/10, Loss: 0.37455274489681417
Epoch 1/10, Loss: 0.1581625289939068
Epoch 2/10, Loss: 0.11544574981118426
Epoch 3/10, Loss: 0.11178914026774987
Epoch 4/10, Loss: 0.10747795107433335
Epoch 5/10, Loss: 0.10624024232105954
Epoch 6/10, Loss: 0.10510669323642557
Epoch 7/10, Loss: 0.10416975198888484
Epoch 8/10, Loss: 0.10395916223280714
Epoch 9/10, Loss: 0.10277759381895693
Accuracy: 0.9426229508196722
Layer 1, Accuracy: 0.9426229508196722
486
544
layer 1 num: 348
Epoch 0/10, Loss: 0.3672997

In [22]:
with open("./edu/parameter_indices.json", 'w') as f:
    json.dump(parameter_indices, f)